# Audio preprocessing

Run the DSP pipeline on generated audio. Set SAMPLE to an authorized WAV or FLAC path for exploratory inspection.

Run setup from a local checkout. Optional installation, downloads and training are off by default.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# In Colab, put an authorized copy of this repository in /content or in Drive.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = Path(os.environ.get("TOPO_LOCK_REPO", ".")).resolve()
if not (ROOT / "pyproject.toml").is_file() and (ROOT.parent / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Set TOPO_LOCK_REPO to the local Acoustic Topo-Lock checkout.")

def run(*args):
    subprocess.run([sys.executable, *args], cwd=ROOT, check=True)

# Set TOPO_LOCK_DATA_DIR and TOPO_LOCK_MODEL_DIR to mounted persistent storage if needed.
INSTALL = False
if INSTALL:
    run("-m", "pip", "install", "-e", ".[research]")

In [ ]:
from acoustic_topo_lock.core.config_loader import load_settings
from acoustic_topo_lock.feature_extraction import extract_features, feature_names
from acoustic_topo_lock.preprocessing import analyze_audio
from acoustic_topo_lock.utils.synthetic_audio import synthetic_wav

SAMPLE = None
settings = load_settings(root=ROOT)
audio = (analyze_audio(SAMPLE, settings.audio) if SAMPLE else
         analyze_audio(synthetic_wav(), settings.audio, filename="software-fixture.wav"))
print(audio.metadata)
print(audio.quality, audio.warnings)
dict(zip(feature_names("room"), extract_features(audio, "room"), strict=True))